# Ascon Side-Channel Analysis: Preprocessing and Deep Learning\n\nThis notebook demonstrates preprocessing of power traces and training a 1D CNN\nto perform a profiled side-channel attack on the Ascon S-box.\n\nAssumes you have already captured traces using `capture_ascon_traces.py` and saved them\nas an HDF5 file (e.g., `ascon_traces.h5`).

## 1. Load and Inspect the Data

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
\n# Load the dataset
filename = 'ascon_traces.h5'  # Update with your file path
with h5py.File(filename, 'r') as f:
    traces = f['traces'][:]
    keys = f['keys'][:]
    plaintexts = f['plaintexts'][:]
    ciphertexts = f['ciphertexts'][:]
\nprint('Traces shape:', traces.shape)
print('Keys shape:', keys.shape)
print('Plaintexts shape:', plaintexts.shape)
print('Ciphertexts shape:', ciphertexts.shape)

## 2. Preprocessing\n\nWe will:\n1. Optionally apply a bandpass filter to remove noise\n2. Align traces using cross-correlation (to reduce jitter)\n3. Select a region of interest (ROI) corresponding to the first S-box operation\n   (this requires knowing approximately where the leakage occurs)\n4. Normalize the traces (e.g., to zero mean and unit variance)

In [ ]:
from scipy import signal
\ndef bandpass_filter(data, lowcut, highcut, fs, order=5):
    """Apply a bandpass filter to the signals."""
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='band')
    y = signal.lfilter(b, a, data, axis=1)
    return y
\ndef align_traces(traces, reference=None, window=None):
    """Align traces using cross-correlation with a reference signal."""
    if reference is None:
        reference = np.mean(traces, axis=0)
    if window is not None:
        start, end = window
        reference = reference[start:end]
    shifts = []
    aligned = np.zeros_like(traces)
    for i, trace in enumerate(traces):
        if window is not None:
            t = trace[start:end]
        else:
            t = trace
        corr = signal.correlate(t, reference, mode='same')
        shift = np.argmax(corr) - len(reference)//2
        shifts.append(shift)
        aligned[i] = np.roll(trace, -shift)
    return aligned, np.array(shifts)
\ndef select_roi(traces, start_idx, end_idx):
    """Select a region of interest (ROI) from the traces."""
    return traces[:, start_idx:end_idx]
\ndef normalize_traces(traces):
    """Normalize traces to zero mean and unit variance."""
    mean = np.mean(traces, axis=1, keepdims=True)
    std = np.std(traces, axis=1, keepdims=True)
    std[std == 0] = 1
    return (traces - mean) / std


### Example Preprocessing Pipeline\n\nNote: The sampling frequency and the expected location of the S-box operation\ndepend on your specific hardware and firmware. You will need to adjust these\nparameters based on your capture settings and knowledge of the device under test.

In [ ]:
# Example parameters - adjust according to your setup
fs = 100e6  # Sampling frequency in Hz (example: 100 MS/s)
lowcut = 0.1e6   # 0.1 MHz
highcut = 20e6   # 20 MHz
\n# Apply bandpass filter
print('Applying bandpass filter...')
filtered_traces = bandpass_filter(traces, lowcut, highcut, fs)
\n# Align traces (using the first 1000 traces as reference for speed)
print('Aligning traces...')
ref_traces = filtered_traces[:min(1000, len(filtered_traces))]
aligned_traces, shifts = align_traces(filtered_traces, reference=np.mean(ref_traces, axis=0))
print('Average shift: {:.2f} samples'.format(np.mean(np.abs(shifts))))
\n# Select ROI - REPLACE WITH YOUR ESTIMATED S-BOX WINDOW
# For example, if the S-box operation occurs around samples 1000-1200:
roi_start = 1000
roi_end = 1200
roi_traces = select_roi(aligned_traces, roi_start, roi_end)
print('ROI shape:', roi_traces.shape)
\n# Normalize
print('Normalizing...')
processed_traces = normalize_traces(roi_traces)


## 3. Create Labels for the S-Box\n\nFor a profiled attack, we need to know the intermediate value we are trying to predict.\nWe will target the output of the first 5-bit S-box in the Ascon initialization phase.\n\nThe Ascon state is 5x64 bits. The first step of the initialization is to XOR the key\nand nonce into the state into state. We can compute the state after the key/nonce\nXOR but before the first round. Then, after the first round's S-box, we get 5 bits per\nword, i.e., 25 bits total. However, a common approach is to attack one 5-bit S-box at a time.\n\nFor simplicity, let's attack the first S-box (least significant 5 bits of the first word).\nWe will predict the Hamming weight (number of 1s) of those 5 bits, which ranges from 0 to 5.

In [ ]:
def ascon_sbox_hw(state_word):
    """Compute the HW of the 5-bit S-box output for a 64-bit word."""
    # Extract 5 least significant bits
    lsb5 = state_word & 0x1F
    # Placeholder S-box: identity (replace with real Ascon S-box)
    sbox_out = lsb5
    return bin(sbox_out).count('1')
\ndef compute_labels(keys, plaintexts):
    """Compute the HW of the first S-box output after key/nonce XOR."""
    labels = []
    for k, p in zip(keys, plaintexts):
        k_int = int.from_bytes(k, byteorder='little')
        p_int = int.from_bytes(p, byteorder='little')
        s0 = k_int ^ p_int
        hw = ascon_sbox_hw(s0)
        labels.append(hw)
    return np.array(labels)
\n# Generate labels
print('Computing labels...')
labels = compute_labels(keys, plaintexts)
print('Labels shape:', labels.shape)
print('Unique labels:', np.unique(labels))
print('Label distribution:')
for i in range(6):
    print('  HW {}: {}'.format(i, np.sum(labels == i)))


## 4. Split the Data\n\nWe will split the data into a training set (for profiling) and a test set (for evaluation).

In [ ]:
from sklearn.model_selection import train_test_split
\n# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    processed_traces, labels, test_size=0.2, random_state=42
)
print('Training set shape:', X_train.shape)
print('Test set shape:', X_test.shape)
print('Training labels shape:', y_train.shape)
print('Test labels shape:', y_test.shape)

## 5. Build and Train a 1D CNN Model\n\nWe will use a simple 1D convolutional neural network.\nThe input is a 1D trace (our ROI), and the output is a probability distribution over\nthe 6 possible Hamming weight values (0-5).

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
\ndef build_model(input_shape):
    model = models.Sequential()
    model.add(layers.Input(shape=input_shape))
    model.add(layers.Conv1D(8, kernel_size=5, activation='relu', padding='same'))
    model.add(layers.AveragePooling1D(pool_size=2))
    model.add(layers.Conv1D(16, kernel_size=5, activation='relu', padding='same'))
    model.add(layers.AveragePooling1D(pool_size=2))
    model.add(layers.Flatten())
    model.add(layers.Dense(32, activation='relu'))
    model.add(layers.Dense(6, activation='softmax'))
    return model
\ninput_shape = (X_train.shape[1], 1)
X_train_reshaped = X_train[..., np.newaxis]
X_test_reshaped = X_test[..., np.newaxis]
\nmodel = build_model(input_shape)
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()


In [ ]:
history = model.fit(
    X_train_reshaped, y_train,
    epochs=20,
    batch_size=100,
    validation_split=0.1,
    verbose=2
)

## 6. Evaluate the Model\n\nWe will evaluate the model on the test set.

In [ ]:
test_loss, test_acc = model.evaluate(X_test_reshaped, y_test, verbose=0)
print('Test accuracy: {:.4f}'.format(test_acc))
\n# Get predictions
y_pred_prob = model.predict(X_test_reshaped)
y_pred = np.argmax(y_pred_prob, axis=1)
\nfrom sklearn.metrics import accuracy_score
acc = accuracy_score(y_test, y_pred)
print('Accuracy from predictions: {:.4f}'.format(acc))
